# Graph Memory: Multi-Hop Reasoning over a Knowledge Graph

### Definition
**Graph Memory** is an agent memory architecture that stores facts as a **knowledge graph** of `(subject, relation, object)` triples instead of flat text chunks or key-value pairs. Each fact becomes an edge connecting two entity nodes. Because facts are explicitly linked, the agent can **traverse** the graph outward from a starting entity to chain multiple facts together and answer **multi-hop questions** — questions whose answer requires combining two or more separate facts that were never stated together in any single sentence.

A flat memory store (a list of text snippets, or a vector index over them) retrieves whichever chunks are *textually similar* to the question. It has no notion of "chase the relation from A to B, then from B to C." Graph memory makes that traversal a first-class operation.

### High-level Workflow

1. **Ingestion (Memory Writer):** Conversational or document text is passed to an LLM configured with `with_structured_output(...)` against a Pydantic schema. The LLM extracts every `(subject, relation, object)` triple it can find and the triples are inserted into the graph store.
2. **Storage:** Triples are stored as directed edges in an in-memory graph — `subject --relation--> object` — indexed by entity so both outgoing and incoming edges can be found quickly.
3. **Query (Memory Reader / QA):** Given a question, an LLM identifies the **starting entity** mentioned in the question. The graph store then performs a bounded **N-hop traversal** outward from that entity, collecting every triple reachable within `max_hops` hops.
4. **Answer Synthesis:** The collected triples (the traversal path) are formatted as context and handed to the LLM, which synthesizes the final answer — with the traversal path itself printed for inspection, so we can see exactly which facts were chained together.

### When to Use / Applications
* **Multi-hop question answering** — "Who manages the person who leads the team that shipped Project X?" style questions that require chaining relations.
* **Entity-centric long-term memory** — assistants that accumulate structured knowledge about people, organizations, and projects across many conversations or documents.
* **Knowledge-base construction** — turning unstructured notes/meeting transcripts into a queryable relationship graph over time.
* **Complement to episodic/semantic memory** — graph memory *is* one concrete implementation of the "semantic memory" half of an episodic+semantic stack (see the Phase 5 pattern demo `11_Advanced_Cognitive_Patterns/03_Episodic_With_Semantic_Memory.ipynb`, which pairs it with a vector store for episodic recall). This notebook goes deeper on the graph piece alone: extraction quality, explicit traversal, and multi-hop QA.

### Strengths & Weaknesses
* **Strengths:**
    * **Multi-hop reasoning:** Can answer questions a flat/unconnected retrieval store structurally cannot, because the connections between facts are preserved, not just the facts themselves.
    * **Explainability:** The exact path of triples used to answer a question can be printed/inspected — useful for debugging and trust.
    * **Deduplication & consolidation:** Facts about the same entity naturally consolidate around one node instead of being scattered across many similar-but-separate text chunks.
* **Weaknesses:**
    * **Extraction is lossy:** Triple extraction depends entirely on the LLM correctly identifying entities and relations; ambiguous coreference ("he", "the team") can silently corrupt the graph.
    * **Schema drift:** Without normalization, the same real-world relation can be written as `works_at`, `employed_by`, `works_for`, fragmenting the graph.
    * **No native fuzzy retrieval:** Unlike a vector store, a graph store needs the query's starting entity to be named (or resolved) correctly, or traversal never begins.
    * **Scale:** A naive in-memory adjacency structure (as used here) doesn't scale to millions of triples — a production system would move to a real graph database (Neo4j, etc.).

### What we are going to do

1. Check the repo's dependencies, then build a tiny **in-memory graph store** (`GraphMemoryStore`) as a dict-of-dicts adjacency structure — no new third-party dependency required, since `networkx` isn't already part of this repo.
2. Define a `Triple` Pydantic schema and a **memory writer** node that uses `llm.with_structured_output(...)` to extract triples from free text and insert them into the store.
3. Define a **memory reader / QA** node that (a) asks the LLM which entity the question is "about", (b) traverses the graph outward from that entity up to `max_hops`, and (c) asks the LLM to answer using only the retrieved triples as context.
4. Feed in a handful of facts spread across *separate* sentences, then ask a **multi-hop** question that requires chaining 2+ triples — printing the exact traversal path so the mechanism is visible, not just the final answer.

## Phase 0: Setup

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from helpers import get_llm

llm = get_llm()

## Phase 1: The Graph Memory Store

We represent the graph as a dict-of-dicts adjacency structure rather than pulling in `networkx` (not currently a dependency of this repo — see `pyproject.toml`). Each node stores its **outgoing** edges (`subject -> [(relation, object), ...]`) and we separately index **incoming** edges so traversal can also follow relations backwards (e.g. "who reports to X" as well as "who does X report to").

* `add_triple(subject, relation, object)` — inserts a directed edge `subject --relation--> object`.
* `query(entity, max_hops=2)` — a breadth-first traversal outward (in both directions) from `entity`, returning every triple reached within `max_hops` hops, along with the hop distance at which it was discovered.

In [ ]:
from dataclasses import dataclass, field


@dataclass
class Triple:
    subject: str
    relation: str
    object: str

    def __str__(self) -> str:
        return f"({self.subject}) --[{self.relation}]--> ({self.object})"


class GraphMemoryStore:
    """A minimal in-memory knowledge graph of (subject, relation, object) triples.

    Stored as a dict-of-dicts adjacency structure:
        _out[entity] = [(relation, other_entity), ...]   outgoing edges
        _in[entity]  = [(relation, other_entity), ...]   incoming edges (for backward traversal)
    """

    def __init__(self) -> None:
        self._out: dict[str, list[tuple[str, str]]] = {}
        self._in: dict[str, list[tuple[str, str]]] = {}
        self._triples: list[Triple] = []

    @staticmethod
    def _norm(entity: str) -> str:
        return entity.strip().lower()

    def add_triple(self, subject: str, relation: str, object: str) -> Triple:
        s, o = self._norm(subject), self._norm(object)
        triple = Triple(subject=subject.strip(), relation=relation.strip(), object=object.strip())
        self._out.setdefault(s, []).append((relation.strip(), o))
        self._in.setdefault(o, []).append((relation.strip(), s))
        # Make sure both endpoints exist as nodes even with no edges in one direction
        self._out.setdefault(o, [])
        self._in.setdefault(s, [])
        self._triples.append(triple)
        return triple

    def _display_name(self, normalized_entity: str) -> str:
        """Best-effort recovery of the original-cased entity name for display."""
        for t in self._triples:
            if self._norm(t.subject) == normalized_entity:
                return t.subject
            if self._norm(t.object) == normalized_entity:
                return t.object
        return normalized_entity

    def query(self, entity: str, max_hops: int = 2) -> list[dict]:
        """Breadth-first traversal outward (both directions) from `entity`.

        Returns a list of dicts: {"hop": int, "triple": Triple, "direction": "out"|"in"}
        capturing every triple discovered within `max_hops` hops of the start entity.
        """
        start = self._norm(entity)
        if start not in self._out and start not in self._in:
            return []

        visited_entities = {start}
        frontier = [start]
        results: list[dict] = []

        for hop in range(1, max_hops + 1):
            next_frontier = []
            for node in frontier:
                for relation, other in self._out.get(node, []):
                    results.append(
                        {
                            "hop": hop,
                            "direction": "out",
                            "triple": Triple(self._display_name(node), relation, self._display_name(other)),
                        }
                    )
                    if other not in visited_entities:
                        visited_entities.add(other)
                        next_frontier.append(other)
                for relation, other in self._in.get(node, []):
                    results.append(
                        {
                            "hop": hop,
                            "direction": "in",
                            "triple": Triple(self._display_name(other), relation, self._display_name(node)),
                        }
                    )
                    if other not in visited_entities:
                        visited_entities.add(other)
                        next_frontier.append(other)
            frontier = next_frontier
            if not frontier:
                break

        return results

    def all_triples(self) -> list[Triple]:
        return list(self._triples)


graph_store = GraphMemoryStore()
print("Graph memory store initialized.")

## Phase 2: The Memory Writer Node

The writer takes a chunk of free text and asks the LLM (via `with_structured_output`) to extract every `(subject, relation, object)` triple it contains. We keep relation names short, lowercase, and verb-like (e.g. `leads`, `manages`, `built`) so the graph stays reasonably normalized.

In [ ]:
from pydantic import BaseModel, Field


class ExtractedTriple(BaseModel):
    subject: str = Field(description="The subject entity of the fact, e.g. a person, team, or project name.")
    relation: str = Field(
        description="A short, lowercase, verb-like relation, e.g. 'leads', 'manages', 'built', 'works_on'."
    )
    object: str = Field(description="The object entity of the fact.")


class ExtractedTriples(BaseModel):
    """All (subject, relation, object) facts found in a piece of text."""

    triples: list[ExtractedTriple] = Field(default_factory=list)


extractor_llm = llm.with_structured_output(ExtractedTriples)


def memory_writer(text: str, store: GraphMemoryStore) -> list[Triple]:
    """Extract triples from `text` with the LLM and insert them into `store`."""
    prompt = (
        "Extract every factual (subject, relation, object) triple stated in the text below. "
        "Use short, lowercase, verb-like relations (e.g. 'leads', 'manages', 'built', 'works_on'). "
        "Use the exact entity names as they appear in the text (people, teams, projects, orgs).\n\n"
        f"Text:\n{text}"
    )
    extracted = extractor_llm.invoke(prompt)
    inserted = []
    for t in extracted.triples:
        inserted.append(store.add_triple(t.subject, t.relation, t.object))
    return inserted

## Phase 3: The Memory Reader / QA Node

Answering a multi-hop question happens in three steps:

1. **Identify the starting entity** — ask the LLM which entity in the graph the question is "about" (its jumping-off point for traversal).
2. **Traverse the graph** — call `store.query(entity, max_hops=...)` to collect every triple within N hops.
3. **Synthesize the answer** — hand the LLM only the retrieved triples (not the whole graph, and not the original source text) and ask it to answer using just that context, so we can verify the answer is actually coming from the traversal.

In [ ]:
class StartEntity(BaseModel):
    entity: str = Field(description="The single entity name in the graph that the question is primarily about.")


entity_picker_llm = llm.with_structured_output(StartEntity)


def memory_reader(question: str, store: GraphMemoryStore, max_hops: int = 2) -> dict:
    """Answer `question` by traversing `store` outward from an LLM-identified start entity."""
    known_entities = sorted(store._out.keys())
    pick_prompt = (
        "Given the question below and this list of known entities in a knowledge graph, "
        "return the single entity name (copied exactly from the list) that the question's "
        "traversal should start from.\n\n"
        f"Known entities: {known_entities}\n\n"
        f"Question: {question}"
    )
    start = entity_picker_llm.invoke(pick_prompt).entity

    traversal = store.query(start, max_hops=max_hops)
    context_lines = [f"hop {r['hop']}: {r['triple']}" for r in traversal]
    context = "\n".join(context_lines) if context_lines else "(no triples found)"

    answer_prompt = (
        "Answer the question using ONLY the facts listed below. Chain multiple facts together "
        "if needed to reach the answer. If the facts are insufficient, say so explicitly.\n\n"
        f"Facts (graph traversal from '{start}'):\n{context}\n\n"
        f"Question: {question}"
    )
    answer = llm.invoke(answer_prompt).content

    return {"start_entity": start, "traversal": traversal, "answer": answer}

## Phase 4: End-to-End Demonstration

We feed in facts about a fictional org across **separate** sentences/documents — no single sentence contains the full chain needed to answer our test question. A flat memory store retrieving by text similarity to the question would likely surface only the sentence that superficially matches the question's wording (about "Project Helios"), and miss the sentence about who manages its team lead. Graph memory instead chains the two facts together via traversal.

In [ ]:
documents = [
    "Project Helios was built by the Atlas team.",
    "Priya Chandran leads the Atlas team.",
    "Priya Chandran reports to Marcus Webb, the VP of Engineering.",
    "Marcus Webb has been VP of Engineering since 2021.",
    "The Atlas team is based in the Bangalore office.",
]

for doc in documents:
    inserted = memory_writer(doc, graph_store)
    print(f"Ingested: {doc!r}")
    for t in inserted:
        print(f"   -> {t}")

In [ ]:
print("All triples currently in the graph:")
for t in graph_store.all_triples():
    print(" ", t)

### Discussion of the Output

Notice that each ingested sentence produced one or more isolated triples: `Project Helios --built_by--> Atlas team`, `Priya Chandran --leads--> Atlas team`, `Priya Chandran --reports_to--> Marcus Webb`, etc. No single document mentions Project Helios and Marcus Webb together — a flat text-similarity retriever asked "who manages the person who leads the team that built Project Helios?" would have no single chunk to return that contains the full answer. Graph memory instead lets us **traverse**: Project Helios -> (built_by) -> Atlas team -> (led_by, reverse of leads) -> Priya Chandran -> (reports_to) -> Marcus Webb.

In [ ]:
multi_hop_question = (
    "Who manages the person who leads the team that built Project Helios?"
)

result = memory_reader(multi_hop_question, graph_store, max_hops=3)

print(f"Start entity identified by LLM: {result['start_entity']!r}\n")
print("Traversal path (triples used as context):")
for r in result["traversal"]:
    print(f"  hop {r['hop']} [{r['direction']}]  {r['triple']}")

print("\nQuestion:", multi_hop_question)
print("Answer:  ", result["answer"])

### Discussion of the Output

The traversal starts from `Project Helios`, hops to `Atlas team` via `built_by`, then to `Priya Chandran` via the reverse of `leads` (found through the incoming-edge index), then to `Marcus Webb` via `reports_to` — three hops chaining three separate facts. The printed traversal path makes the reasoning **inspectable**: we can see exactly which edges were walked to reach the answer, rather than trusting an opaque retrieval score. Try lowering `max_hops` to `1` and re-running — the traversal will stop at `Atlas team`/`Priya Chandran` and the LLM should explicitly say the facts are insufficient to name a manager, which demonstrates why multi-hop traversal (not single-hop lookup) is the important capability here.

## Summary & Key Takeaways

* **Graph memory** stores facts as `(subject, relation, object)` triples in a directed graph rather than flat text chunks, making the *connections* between facts a first-class, queryable structure.
* A **memory writer** node uses `llm.with_structured_output(...)` to turn unstructured text into triples; a **memory reader** node identifies a starting entity, performs a bounded-hop traversal, and answers using only the retrieved triples — with the traversal path fully inspectable.
* This directly enables **multi-hop question answering** ("who manages the person who leads the team that built X?") that a flat/unconnected memory store cannot answer, because the required facts were never stated together in one place.
* This is one of several **memory architecture** patterns collected under `07_Advanced_Agentic_Systems/Memory_and_State/Agentic_Memory_Architectures/` (a new sibling to the framework-mechanics-focused `LangChain/` and `LangGraph/` folders). Related notebooks being authored separately in this same folder cover **MemGPT-style tiered memory** (OS-inspired paging between working/long-term memory), the **Voyager skill library** (accumulating reusable skills instead of facts), and **Agent Workflow Memory** (memory of successful *procedures*, not just facts) — together these span the space of what an agent can remember and how it decides what to keep, forget, or re-use.
* **Production caveat:** the in-memory dict-of-dicts store here is for teaching the mechanism clearly. A real system would use a proper graph database (Neo4j, etc.), add entity resolution/normalization to avoid relation-name drift (`works_at` vs `employed_by`), and add pruning/consolidation strategies as the graph grows.